In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torchinfo import summary

In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, x):
        return self.out(F.relu(self.hidden(x)))  # 神经层由内向外衔接

In [3]:
net = MLP()
x=torch.rand(2,20)  # rand[0,1)均匀分布 randn正态分布
net(x)
summary(net,input_size=(2,20))

tensor([[ 0.0089, -0.0897,  0.2309, -0.0129,  0.1455, -0.1271, -0.0234, -0.0553,
          0.0431, -0.0455],
        [ 0.0158,  0.0100,  0.2319, -0.1331,  0.0774, -0.0516,  0.0769, -0.1224,
          0.0822,  0.0373]], grad_fn=<AddmmBackward0>)

In [4]:
print(net(x))

tensor([[ 0.0089, -0.0897,  0.2309, -0.0129,  0.1455, -0.1271, -0.0234, -0.0553,
          0.0431, -0.0455],
        [ 0.0158,  0.0100,  0.2319, -0.1331,  0.0774, -0.0516,  0.0769, -0.1224,
          0.0822,  0.0373]], grad_fn=<AddmmBackward0>)


In [5]:
# 顺序块
class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for idx,module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self,X):
        for block in self._modules.values():
            X = block(X)
        return X

In [6]:
net = MySequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net(x)

tensor([[ 0.0949, -0.1709,  0.0808,  0.0774, -0.0153,  0.0296, -0.1927, -0.2420,
         -0.2473,  0.1353],
        [ 0.0350, -0.1525, -0.0236,  0.0497, -0.0829, -0.0667, -0.1420, -0.2553,
         -0.1459,  0.1673]], grad_fn=<AddmmBackward0>)

In [7]:
print(net(x))

tensor([[ 0.0949, -0.1709,  0.0808,  0.0774, -0.0153,  0.0296, -0.1927, -0.2420,
         -0.2473,  0.1353],
        [ 0.0350, -0.1525, -0.0236,  0.0497, -0.0829, -0.0667, -0.1420, -0.2553,
         -0.1459,  0.1673]], grad_fn=<AddmmBackward0>)


In [8]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20,20),requires_grad=False)  # 这个参数设置常量
        self.linear = nn.Linear(20,20)

    def forward(self, x):
        x = self.linear(x)
        x = F.relu(torch.mm(x,self.rand_weight)+1)  # mm是矩阵运算
        x = self.linear(x)
        while x.abs().sum() > 1:
            x /= 2
        return x.sum()

In [9]:
net = FixedHiddenMLP()
net(x)
summary(net, input_size=(2, 20))

Layer (type:depth-idx)                   Output Shape              Param #
FixedHiddenMLP                           --                        --
├─Linear: 1-1                            [2, 20]                   420
├─Linear: 1-2                            [2, 20]                   (recursive)
Total params: 420
Trainable params: 420
Non-trainable params: 0
Total mult-adds (M): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [10]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU())  #激活函数只作用前一层的输出
        self.linear = nn.Linear(32,16)

    def forward(self, x):
        return self.linear(self.net(x))

chimera = nn.Sequential(NestMLP(),nn.Linear(16,20),FixedHiddenMLP())  # 顺序层级
print(chimera(x))

tensor(-0.2977, grad_fn=<SumBackward0>)


In [11]:
for name, module in chimera.named_modules():
    print(name, "->", module)

 -> Sequential(
  (0): NestMLP(
    (net): Sequential(
      (0): Linear(in_features=20, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
      (3): ReLU()
    )
    (linear): Linear(in_features=32, out_features=16, bias=True)
  )
  (1): Linear(in_features=16, out_features=20, bias=True)
  (2): FixedHiddenMLP(
    (linear): Linear(in_features=20, out_features=20, bias=True)
  )
)
0 -> NestMLP(
  (net): Sequential(
    (0): Linear(in_features=20, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
  )
  (linear): Linear(in_features=32, out_features=16, bias=True)
)
0.net -> Sequential(
  (0): Linear(in_features=20, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
)
0.net.0 -> Linear(in_features=20, out_features=64, bias=True)
0.net.1 -> ReLU()
0.net.2 -> Linear(in_features=64, out_features=32, bias

In [12]:
summary(chimera, input_size=(2, 20))  # 打印树状图

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               --                        --
├─NestMLP: 1-1                           [2, 16]                   --
│    └─Sequential: 2-1                   [2, 32]                   --
│    │    └─Linear: 3-1                  [2, 64]                   1,344
│    │    └─ReLU: 3-2                    [2, 64]                   --
│    │    └─Linear: 3-3                  [2, 32]                   2,080
│    │    └─ReLU: 3-4                    [2, 32]                   --
│    └─Linear: 2-2                       [2, 16]                   528
├─Linear: 1-2                            [2, 20]                   340
├─FixedHiddenMLP: 1-3                    --                        --
│    └─Linear: 2-3                       [2, 20]                   420
│    └─Linear: 2-4                       [2, 20]                   (recursive)
Total params: 4,712
Trainable params: 4,712
Non-trainable params: 0